[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BioImagingUK/Training/blob/main/notebooks/Analysis.ipynb)

## Learning objectives

* Read data to analyse using [BioIO](https://github.com/bioio-devs/bioio).
* Analyse the data sequentially.
* Analyse the data in parallel.


**BioIO**

In order to analyse your image data, you need to be able to read the numerous proprietary file formats (PFFs). Bio-Formats is the most comprehensive tool in Java to read PFFs, but no such library exists in Python yet.

Nonetheless the [BioIO](https://github.com/bioio-devs/bioio) library is a more domain-specific Python library for accessing data. It wraps several libraries for microscopy vendor's formats (including nd2, czi) and has a Python wrapper around the reference Java library for reading many proprietary file formats Bio-Formats. This is obviously not a pure Python solution since it requires Java installed in order to run.

### Download an image to analyse.

In [1]:
!wget https://downloads.openmicroscopy.org/images/ND2/aryeh/MeOh_high_fluo_011.nd2

### Install the BioIO dependencies

In [3]:
%pip install bioio
%pip install bioio-nd2

### Read the image

In [56]:
from skimage.filters import threshold_otsu
from skimage.morphology import closing
from skimage.filters import gaussian
from skimage.measure import label

def analyze(t, c, z):
    plane = data[t, c, z, :, :]
    black_white_plane = closing(plane > threshold_otsu(plane))
    label_image = label(black_white_plane)
    name = "t:%s, c: %s, z:%s" % (t, c, z)
    return label_image, name

In [9]:
from bioio import BioImage
import bioio_nd2

# Adjust the local path to the image
image = BioImage("/content/MeOh_high_fluo_011.nd2", reader=bioio_nd2.Reader)
print(image.dims.order) # Return the dimension order
print(image.shape)

Import dependencies to display the results

In [54]:
import matplotlib.pyplot as plt
%matplotlib inline

### Analyse the image
Analyse each timepoint sequentially.


In [57]:
%%time
data = image.data
size_t = image.shape[0]
for t in range(size_t):
    plane = data[t, 0, 0, :, :]
    label_image, name = analyze(t, 0, 0)
    fig = plt.figure(figsize=(10, 10))
    sub1 = plt.subplot(121)
    sub1.title.set_text(f'Original Image {name}')
    plt.imshow(plane)
    sub2 = plt.subplot(122)
    sub2.title.set_text(f'Label Image {name}')
    plt.imshow(label_image)
    plt.tight_layout()
    fig.canvas.flush_events()


### Analyse in parallel
The BioIO library allows to read the imaging data using [Dask array](https://docs.dask.org/en/stable/array.html). This strategy allows use to analyse data in parallel.

In [ ]:
import dask

Load the image as a Dask array. In that case, the binary data is not loaded. It is an "empty container"

In [59]:
data_lazy = image.dask_data
print(data_lazy.shape)

(13, 1, 1, 600, 800)


Define the analysis method. The binary data is loaded when the ``compute()`` method is invoked.



In [48]:
def analyze_parallel(t):
    plane = data_lazy[t, 0, 0, :, :].compute()
    black_white_plane = closing(plane > threshold_otsu(plane))
    label_image = label(black_white_plane)
    name = "t:%s, c: %s, z:%s" % (t, 0, 0)
    return label_image, name

Build the analysis graph.

In [49]:
%%time
def build_task_graph():
    lazy_results = []
    for t in range(size_t):
        lazy_result = dask.delayed(analyze_parallel)(t)
        lazy_results.append(lazy_result)
    return lazy_results

In [50]:
lazy_results = build_task_graph()
print(lazy_results)

Execute the analysis.

In [51]:
%%time
# Analyse the data in parallel
results = dask.compute(*lazy_results)

Plot the results

In [58]:
for i in range(len(results)):
    r, name = results[i]
    fig = plt.figure(figsize=(10, 10))
    plt.subplot(121)
    plt.imshow(r)
    plt.title(f"{name}")
    fig.canvas.flush_events()

License (BSD 2-Clause)
Copyright (C) 2026 BioImagingUK. All Rights Reserved.

Redistribution and use in source and binary forms, with or without modification, are permitted provided that the following conditions are met:

Redistributions of source code must retain the above copyright notice, this list of conditions and the following disclaimer. Redistributions in binary form must reproduce the above copyright notice, this list of conditions and the following disclaimer in the documentation and/or other materials provided with the distribution. THIS SOFTWARE IS PROVIDED BY THE COPYRIGHT HOLDERS AND CONTRIBUTORS "AS IS" AND ANY EXPRESS OR IMPLIED WARRANTIES, INCLUDING, BUT NOT LIMITED TO, THE IMPLIED WARRANTIES OF MERCHANTABILITY AND FITNESS FOR A PARTICULAR PURPOSE ARE DISCLAIMED. IN NO EVENT SHALL THE COPYRIGHT OWNER OR CONTRIBUTORS BE LIABLE FOR ANY DIRECT, INDIRECT, INCIDENTAL, SPECIAL, EXEMPLARY, OR CONSEQUENTIAL DAMAGES (INCLUDING, BUT NOT LIMITED TO, PROCUREMENT OF SUBSTITUTE GOODS OR SERVICES; LOSS OF USE, DATA, OR PROFITS; OR BUSINESS INTERRUPTION) HOWEVER CAUSED AND ON ANY THEORY OF LIABILITY, WHETHER IN CONTRACT, STRICT LIABILITY, OR TORT (INCLUDING NEGLIGENCE OR OTHERWISE) ARISING IN ANY WAY OUT OF THE USE OF THIS SOFTWARE, EVEN IF ADVISED OF THE POSSIBILITY OF SUCH DAMAGE.